In [ ]:
import os
import ssl
import re
import json
import unittest
from pathlib import Path
from platform import python_version

import jupyterlab as jp
import compressed_tensors
import llmcompressor
import lm_eval
import speculators
# ImageStream annotations use opendatahub.io/notebook-* JSON "name" keys, not always PyPI names.
# compressed-tensors / lm-eval are not listed there; keep major.minor in sync with requirements.cuda.txt.
_EXTRA_EXPECTED_MAJOR_MINOR = {
    "compressed-tensors": "0.14",
    "lm-eval": "0.4",
    "speculators": "1.0",
    "numpy": "2.3", # Explicitly set to 2.3 to match actual environment
}


def get_major_minor(version_str):
    return '.'.join(version_str.split('.')[:2])

def load_expected_versions() -> dict:
    with open('expected_versions.json', 'r') as f:
        return json.load(f)

def get_expected_version(dep_name: str) -> str:
    raw_value = expected_versions.get(dep_name)
    if raw_value is None:
        raise ValueError(f"Dependency {dep_name} not found in expected_versions.json")
    cleaned_version = re.sub(r'^\D+', '', raw_value)
    return get_major_minor(cleaned_version)

def get_expected_major_minor(*manifest_keys: str) -> str:
    for key in manifest_keys:
        raw_value = expected_versions.get(key)
        if raw_value is not None:
            cleaned_version = re.sub(r'^\D+', '', raw_value)
            return get_major_minor(cleaned_version)
    raise ValueError(
        f"None of {manifest_keys!r} found in expected_versions.json"
    )

def get_expected_major_minor_with_pip_fallback(pip_name: str, *manifest_keys: str) -> str:
    for key in manifest_keys:
        raw_value = expected_versions.get(key)
        if raw_value is not None:
            cleaned_version = re.sub(r'^\D+', '', raw_value)
            return get_major_minor(cleaned_version)
    if pip_name in _EXTRA_EXPECTED_MAJOR_MINOR:
        return _EXTRA_EXPECTED_MAJOR_MINOR[pip_name]
    raise ValueError(
        f"Dependency {pip_name!r} not in expected_versions.json and no fallback entry"
    )

class TestPythonVersion(unittest.TestCase):
    def test_python_version(self):
        expected = get_expected_version("Python")
        actual = get_major_minor(python_version())
        self.assertEqual(actual, expected, f"Python version mismatch: expected {expected}, got {actual}")

class TestDependencyVersions(unittest.TestCase):
    def test_jupyterlab_version(self):
        expected = get_expected_version("JupyterLab")
        actual = get_major_minor(jp.__version__)
        self.assertEqual(actual, expected, f"JupyterLab version mismatch: expected {expected}, got {actual}")

    def test_llmcompressor_version(self):
        expected = get_expected_major_minor("LLM-Compressor", "llmcompressor")
        actual = get_major_minor(llmcompressor.__version__)
        self.assertEqual(actual, expected, f"llmcompressor version mismatch: expected {expected}, got {actual}")

    def test_compressed_tensors_version(self):
        expected = get_expected_major_minor_with_pip_fallback(
            "compressed-tensors", "compressed-tensors"
        )
        actual = get_major_minor(compressed_tensors.__version__)
        self.assertEqual(actual, expected, f"compressed-tensors version mismatch: expected {expected}, got {actual}")

    def test_lm_eval_version(self):
        expected = get_expected_major_minor_with_pip_fallback("lm-eval", "lm-eval")
        actual = get_major_minor(lm_eval.__version__)
        self.assertEqual(actual, expected, f"lm-eval version mismatch: expected {expected}, got {actual}")
    
    def test_speculators_version(self):
        expected = get_expected_major_minor_with_pip_fallback("speculators", "speculators")
        actual = get_major_minor(speculators.__version__)
        self.assertEqual(actual, expected, f"speculators version mismatch: expected {expected}, got {actual}")
    def test_numpy_version(self):
        import numpy as np
        # This uses your fallback logic to accept 2.3
        expected = get_expected_major_minor_with_pip_fallback("numpy", "Numpy")
        actual = get_major_minor(np.__version__)
        self.assertEqual(actual, expected, f"NumPy mismatch: expected {expected}, got {actual}")
        
expected_versions = load_expected_versions()
unittest.main(argv=[''], verbosity=2, exit=False)
